In [3]:
import os 
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
               "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
               "the-verdict.txt")
    file_path="the-verdict.txt"
    urllib.request.urlretrieve(url,file_path)
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()
import tiktoken 
from torch.utils.data import Dataset,DataLoader
import torch
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
   self.inputs_ids=[]
   self.target_ids=[]
   token_ids=tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
   for i in range(0,len(token_ids)-max_length,stride):
     input_chunks=token_ids[i:i+max_length]
     target_chunks=token_ids[i+1:i+max_length+1]
     self.inputs_ids.append(torch.tensor(input_chunks))
     self.target_ids.append(torch.tensor(target_chunks))
  def __len__(self):
     return len(self.inputs_ids)
  def __getitem__(self, index):
    return self.inputs_ids[index],self.target_ids[index]
def create_dataloader_v1(txt,batch_size=2,max_length=256 ,stride=128, shuffle=True ,drop_last=True,num_worker=0):
   tokenizer = tiktoken.get_encoding("gpt2")
   dataset =GPTDatasetV1(txt,tokenizer,max_length=4,stride=128)
   dataloader=DataLoader( dataset, 
                       batch_size=batch_size,
                       shuffle=shuffle,
                       drop_last=drop_last,
                       num_workers=num_worker      )
   return dataloader
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter=iter(dataloader)
inputs,target=next(data_iter)
# print("inputs\n",inputs)
# print("target\n",target)
vocab_size=50257
output_dim=4
torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
token_embedding=embedding_layer(inputs)
max_length=4
pos_embedding_layer=torch.nn.Embedding(max_length,output_dim)
pos_embedding=pos_embedding_layer(torch.arange(max_length))
input_embedding=token_embedding+pos_embedding
# inputs_q=token_embedding[0][1]
# inputs_test=token_embedding[0]
# p=torch.empty(inputs_test.shape[0])
# for i,elem in enumerate(inputs_test):
#      p[i]=torch.dot(elem,inputs_q)
# print(torch.softmax(p,dim=0))


In [4]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [5]:
inputs_q=inputs[1]
p=torch.empty(inputs.shape[0])
for i,elem in enumerate(inputs):
     p[i]=torch.dot(elem,inputs_q)
attn_weigth_2=torch.softmax(p,dim=0)
print(p)
print(attn_weigth_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [6]:
context_vec_2=torch.zeros(inputs_q.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+= attn_weigth_2[i]*inputs[i]
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [7]:
dot_product=torch.empty(6,6)
for i ,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        dot_product[i,j]=torch.dot(x_i,x_j)
print(dot_product)


tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [9]:
assign=inputs @ inputs.T
# print(assign)
assign_weight= torch.softmax(assign,dim=1)
print(assign_weight )
all_context_wegiht=assign_weight @ inputs
print(all_context_wegiht)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
